In [1]:
from transformers import pipeline
import torch
import pandas as pd
import numpy as np
import re


c:\Users\pavel\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("Data/Articles/articles_content.csv")
df = df.dropna()
df
keywords = pd.read_csv("Data/Articles/financial_keywords.csv")
keywords = keywords["0"].to_list()

In [3]:
def get_summary(df, content_column_name: str = "Content", use_gpu: bool = False, save_csv: bool = False):

    df = df.copy()

    if use_gpu:
        comp = 0
    else:
        comp = 'cpu'

    content_list = df[content_column_name].to_list()

    number_of_articles = len(content_list)

    model_sum = pipeline("summarization", model="facebook/bart-large-cnn", framework="pt", device=comp)
    model_par = pipeline("text2text-generation", model="facebook/bart-large", framework="pt", device=comp)

    summaries = []

    for i, content in enumerate(content_list):

        print(f"Extraction process: {i + 1}/{number_of_articles}")

        content_len = len(content)
        if content_len > 8600:
            summaries.append(np.nan)
            continue
        
        if content_len > 4200:
            first_part = content[:int(content_len/2)]
            index = first_part.rindex(".")
            first_part = content[:index+1]
            last_part = content[index+1:]

            summary_1 = model_sum(first_part, max_length=50, min_length=30, do_sample=False, truncation=True)
            summary_2 = model_sum(last_part, max_length=50, min_length=30, do_sample=False, truncation=True)

            summary_1 = summary_1[0]["summary_text"]
            summary_2 = summary_2[0]["summary_text"]

            meta_summary = summary_1 + summary_2

            summary =  model_par(meta_summary, max_length = 120, min_length=50, do_sample=False, truncation=True)

            summaries.append(summary[0]["generated_text"])

        else:
            summary = model_sum(content, max_length=120, min_length=70, do_sample=False, truncation=True)
            summaries.append(summary[0]["summary_text"])

    df["summary"] = summaries

    if save_csv:
        df.to_csv("Data/Articles/articles_with_summary.csv", index=False)
            
    return df
        
        

In [4]:
def get_sentiment(df, content_column_name: str = "Content", use_gpu: bool = False, save_csv: bool = False):

    df = df.copy()

    if use_gpu:
        comp = 0
    else:
        comp = 'cpu'

    content_list = df[content_column_name].to_list()
    model_sen = pipeline("text-classification", model="ProsusAI/finbert", framework="pt", top_k=3, device=comp)

    sentiment_scores = model_sen(content_list, truncation= True)

    sentiments = []
    for scores in sentiment_scores:
        for score in scores:
            if score["label"] == "positive":
                score_pos = score["score"]
            elif score["label"] == "negative":
                score_neg = score["score"]
        sentiments.append(score_pos - score_neg)

    df["Sentiment"] = sentiments

    
    if save_csv:
        df.to_csv("Data/Articles/articles_with_sentiment.csv", index=False)

    return df



In [5]:
def get_relevance(df, keywords: list, content_column_name: str = "Content", save_csv: bool = False):

    df = df.copy()
    content_list = df[content_column_name].to_list()

    relevance_list = []
    number_of_articles = len(content_list)

    special_signs = [".", "?", "!", "\\", ",", '"', "-"]

    for i,content in enumerate(content_list):

        print(f"Relevance process: {i + 1}/{number_of_articles}")
        

        for sign in special_signs:
            content = content.replace(sign, " ")
            content = content.lower()

        for keyword in keywords:
            content = content.replace(keyword, keyword.replace(" ", "_"))
        
        keywords_counter = {keyword.replace(" ", "_").lower(): 0 for keyword in keywords}

        all_words = re.findall(r'\b\w+\b', content)
        for word in all_words:
            if word in keywords_counter.keys():
                keywords_counter[word] += 1


        relevance_list.append(sum(keywords_counter.values()) / len(all_words))

    relevance_list = [i / max(relevance_list) for i in relevance_list]
    
    df["Relevance"] = relevance_list
    
    if save_csv:
        df.to_csv("Data/Articles/articles_with_relevance.csv", index=False)
        
    return df




In [6]:
def get_article_info(df: pd.DataFrame, keywords: list, content_column_name: str = "Content", use_gpu: bool = False, save_summary: bool = False, save_sentiment: bool = False, save_relevance: bool = False, save_result: bool = True):
    df = get_summary(df, use_gpu=use_gpu, save_csv=save_summary)
    df = get_sentiment(df, use_gpu=use_gpu, save_csv=save_sentiment)
    df = get_relevance(df, keywords=keywords, save_csv=save_relevance)

    if save_result:
        df.to_csv("Data/Articles/article_result.csv")
    return df


In [ ]:
df = get_article_info(df, keywords, use_gpu=True, save_result=True)
df.head(10)

,Date,Title,Content,summary,Sentiment,Relevance
0,2025-04-10,Is the US making $2bn a day from tariffs? Trum...,President Donald Trump has announced sweeping ...,President Donald Trump has announced a 90-day ...,-0.069009,0.194872
1,2025-04-09,How exposed is the UK to Trump's tariff chaos?,Warnings of a global recession have been inten...,"US tariffs hit the UK, China harderThe Trump a...",-0.655420,0.390282
3,2025-04-09,Trump rips up rulebook on trade and businesses...,US President Donald Trump is ripping up the ru...,US President Donald Trump is ripping up the ru...,-0.658769,0.174502
5,2025-03-26,UK inflation rate: How quickly are prices rising?,Prices in the UK rose by 2.8% in the 12 months...,"CPI was 2.8% in year to February 2025, down fr...",-0.922021,0.489202
6,2025-03-20,Fed cuts US growth forecast as tariff fears in...,The US central bank has cut its growth forecas...,The Federal Reserve has cut its growth forecas...,-0.910005,0.446894
7,2025-03-19,Splurge or save? Americans struggle as tariffs...,A few days after Donald Trump won the US presi...,US families are cutting back on spending as ta...,-0.887614,0.320888
8,2025-03-13,US tech firms feel pinch from China tariffs,Deena Ghazarian had only been in business for ...,Deena Ghazarian had only been in business for ...,-0.165571,0.079931
9,2025-03-12,Is the US really heading into a recession?,"During his election campaign last year, Donald...",The US economy was already undergoing a slowdo...,-0.891317,0.390038
10,2025-03-11,Trump is no longer swayed by the stock markets,On the day of the US presidential inauguration...,President Trump has said he is rebuilding weal...,-0.436687,0.451616
11,2025-03-07,US job growth stable as government cuts start,US President Donald Trump's cuts to the govern...,"US employers added 151,000 jobs, while the une...",-0.883173,0.276999
